# Pre-Processing for logs that are not in .xes.gz format

Converts CSV sources (and a BPIC12 smoke-test sample) to `.xes.gz` under `data/real/`.

Run with the project root as the working directory, or from `analyses/` (paths resolve via `project_root`).

To verify that this notebook reproduces the working logs without overwriting them, set `OUTPUT_INFIX` (e.g. `".verify"`) so outputs become `NAME.verify.xes.gz`, then run the comparison cell at the end.


## Setup

In [ ]:
import hashlib
import gzip
import os
from pathlib import Path

import pandas as pd
from pm4py.objects.log.util import dataframe_utils
from pm4py.objects.conversion.log import converter as log_converter
from pm4py.objects.log.exporter.xes import exporter as xes_exporter

# Resolve project root whether cwd is the repo root or analyses/.
cwd = Path.cwd().resolve()
if (cwd / "data" / "real").is_dir():
    project_root = cwd
elif (cwd.parent / "data" / "real").is_dir():
    project_root = cwd.parent
else:
    raise FileNotFoundError(f"Could not locate data/real from cwd={cwd}")

def data_path(*parts: str) -> Path:
    return project_root.joinpath("data", *parts)

# Inserted before ".xes.gz". Empty string writes the working filenames.
# Example: OUTPUT_INFIX = ".verify" -> ACCRE.verify.xes.gz
OUTPUT_INFIX = ".verify"

def xes_out(dataset_dir: str, stem: str) -> Path:
    """Build output path: data/real/<dataset_dir>/<stem><OUTPUT_INFIX>.xes.gz"""
    return data_path("real", dataset_dir, f"{stem}{OUTPUT_INFIX}.xes.gz")

print("project_root =", project_root)
print("OUTPUT_INFIX =", repr(OUTPUT_INFIX))


## Test log

Create a **500**-trace sample from BPIC 2012 to test the scripts (not to be used in formal evaluations).


In [ ]:
from pm4py.objects.log.importer.xes import importer as xes_importer
from pm4py.objects.log.obj import EventLog

log_path = data_path("real", "BPIC12", "BPI_Challenge_2012.xes.gz")
out_path = xes_out("TEST_BPIC12", "TEST_BPI_Challenge_2012")

log = xes_importer.apply(str(log_path))
number_traces = 500
sampled_log = EventLog(log[:number_traces])
os.makedirs(out_path.parent, exist_ok=True)
xes_exporter.apply(sampled_log, str(out_path))
print("Wrote", out_path, "with", number_traces, "traces")


## ACCRE-log

The log has been split into test and train. Each contains only completed traces. Hence, some traces that start in the train data and finish in the test data have been removed. Because we want to avoid detecting drifts due to these removed traces, we only include the train dataset.


In [ ]:
log_path = data_path("real", "ACCRE", "AcademicCredentials_train.csv.gz")
df = pd.read_csv(log_path, compression="gzip")

df["case:concept:name"] = df["case_id"]
df["concept:name"] = df["activity"]
df["time:timestamp"] = df["start_time"]
df["org:resource"] = df["resource"]

df = dataframe_utils.convert_timestamp_columns_in_df(df)
df = df.sort_values(by="time:timestamp").reset_index(drop=True)

event_log = log_converter.apply(df, variant=log_converter.Variants.TO_EVENT_LOG)
output_path = xes_out("ACCRE", "ACCRE")
xes_exporter.apply(event_log, str(output_path), parameters={"gzip": True})
print("Wrote", output_path, "| events:", len(df), "| traces:", df["case:concept:name"].nunique())


## BPIC14

In [ ]:
log_path = data_path("real", "BPIC14", "Detail_Incident_Activity.csv.gz")
df = pd.read_csv(log_path, delimiter=";", compression="gzip")

df["case:concept:name"] = df["Incident ID"]
df["concept:name"] = df["IncidentActivity_Type"]
df["time:timestamp"] = df["DateStamp"]

df = dataframe_utils.convert_timestamp_columns_in_df(df)
df = df.sort_values(by="time:timestamp").reset_index(drop=True)

event_log = log_converter.apply(df, variant=log_converter.Variants.TO_EVENT_LOG)
output_path = xes_out("BPIC14", "BPIC14")
xes_exporter.apply(event_log, str(output_path), parameters={"gzip": True})
print("Wrote", output_path, "| events:", len(df), "| traces:", df["case:concept:name"].nunique())


## CALL

The log has been split into test and train. Each contains only completed traces. Hence, some traces that start in the train data and finish in the test data have been removed. Because we want to avoid detecting drifts due to these removed traces, we only include the train dataset.


In [ ]:
log_path = data_path("real", "CALL", "CallCenter_train.csv.gz")
df = pd.read_csv(log_path, delimiter=",")

df["case:concept:name"] = df["case_id"]
df["concept:name"] = df["activity"]
df["time:timestamp"] = df["end_time"]
df["org:resource"] = df["resource"]

df = dataframe_utils.convert_timestamp_columns_in_df(df)
df = df.sort_values(by="time:timestamp").reset_index(drop=True)

event_log = log_converter.apply(df, variant=log_converter.Variants.TO_EVENT_LOG)
output_path = xes_out("CALL", "CALL")
xes_exporter.apply(event_log, str(output_path), parameters={"gzip": True})
print("Wrote", output_path, "| events:", len(df), "| traces:", df["case:concept:name"].nunique())


## ITHD

In [ ]:
log_path = data_path("real", "ITHD", "finale.csv.gz")
df = pd.read_csv(log_path, delimiter=",")

df["case:concept:name"] = df["Case ID"]
df["concept:name"] = df["Activity"]
df["time:timestamp"] = df["Complete Timestamp"]
df["org:resource"] = df["Resource"]

df = dataframe_utils.convert_timestamp_columns_in_df(df)
df = df.sort_values(by="time:timestamp").reset_index(drop=True)

event_log = log_converter.apply(df, variant=log_converter.Variants.TO_EVENT_LOG)
output_path = xes_out("ITHD", "ITHD")
xes_exporter.apply(event_log, str(output_path), parameters={"gzip": True})
print("Wrote", output_path, "| events:", len(df), "| traces:", df["case:concept:name"].nunique())


## MOBIS

In [ ]:
log_path = data_path("real", "MOBIS", "10258_mobis_challenge_log_2019.csv.gz")
df = pd.read_csv(log_path, delimiter=";")

df["case:concept:name"] = df["case"]
df["concept:name"] = df["activity"]
df["time:timestamp"] = df["end"]

df["time:timestamp"] = pd.to_datetime(
    df["time:timestamp"],
    format="%d.%m.%Y %H:%M",
    utc=True,
)
df = df.sort_values(by="time:timestamp").reset_index(drop=True)

event_log = log_converter.apply(df, variant=log_converter.Variants.TO_EVENT_LOG)
output_path = xes_out("MOBIS", "MOBIS")
xes_exporter.apply(event_log, str(output_path), parameters={"gzip": True})
print("Wrote", output_path, "| events:", len(df), "| traces:", df["case:concept:name"].nunique())


## Compare regenerated outputs to working logs

Compares each `stem{OUTPUT_INFIX}.xes.gz` against the working `stem.xes.gz` in the same folder.

- **gzip_identical**: byte-identical compressed files
- **raw_xes_identical**: identical after gunzip (ignores gzip wrapper/metadata)
- If PM4Py re-export changes XML formatting, also report trace/event counts from both files.


In [ ]:
from pm4py.objects.log.importer.xes import importer as xes_importer

pairs = [
    ("ACCRE", "ACCRE.xes.gz"),
    ("BPIC14", "BPIC14.xes.gz"),
    ("CALL", "CALL.xes.gz"),
    ("ITHD", "ITHD.xes.gz"),
    ("MOBIS", "MOBIS.xes.gz"),
    ("TEST_BPIC12", "TEST_BPI_Challenge_2012.xes.gz"),
]

def md5_bytes(data: bytes) -> str:
    return hashlib.md5(data).hexdigest()

def md5_file(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def md5_gunzip(path: Path) -> str:
    h = hashlib.md5()
    with gzip.open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if not OUTPUT_INFIX:
    print("OUTPUT_INFIX is empty — skipping comparison (would compare a file to itself).")
else:
    rows = []
    for dataset, working_name in pairs:
        working = data_path("real", dataset, working_name)
        stem = working_name[: -len(".xes.gz")]
        regenerated = data_path("real", dataset, f"{stem}{OUTPUT_INFIX}.xes.gz")
        if not regenerated.is_file():
            rows.append((dataset, "MISSING_REGEN", "-", "-", "-", "-"))
            continue
        if not working.is_file():
            rows.append((dataset, "MISSING_WORKING", "-", "-", "-", "-"))
            continue
        gz_eq = md5_file(working) == md5_file(regenerated)
        raw_eq = md5_gunzip(working) == md5_gunzip(regenerated)
        if raw_eq:
            n_w = n_r = "n/a (raw match)"
        else:
            wlog = xes_importer.apply(str(working))
            rlog = xes_importer.apply(str(regenerated))
            n_w = f"{len(wlog)} traces / {sum(len(t) for t in wlog)} events"
            n_r = f"{len(rlog)} traces / {sum(len(t) for t in rlog)} events"
        rows.append((dataset, "ok", gz_eq, raw_eq, n_w, n_r))

    print(f"{'dataset':12} {'status':16} {'gz_eq':6} {'raw_eq':6} working_counts | regenerated_counts")
    for dataset, status, gz_eq, raw_eq, n_w, n_r in rows:
        print(f"{dataset:12} {status:16} {str(gz_eq):6} {str(raw_eq):6} {n_w} | {n_r}")
